# BRMARL v2 — frozen holdout и отдельная online-адаптация

Каждый режим заново загружает один checkpoint. Frozen не меняет параметры; online обновляет BRMARL внутри
смены, используя предобученные context/OSR/CFR, фиксированные до конца этой смены. Между online-сменами
разрешены послесменные обновления. Истинные клиентские theta доступны только механизму аукциона.

In [1]:
from pathlib import Path
import os, sys, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, IFrame
ROOT=Path.cwd()
assert (ROOT/'brmarl').is_dir(), 'Запустите notebook из распакованного проекта'
sys.path.insert(0,str(ROOT))
from brmarl.config import MarketConfig, AgentConfig, ParallelConfig
from brmarl.data import (load_assets, make_fixture_assets, prepare_route_cache,
                         create_search_index, prewarm_centroid_pool)
from brmarl.experiment import train, test, load_checkpoint
from brmarl.evaluation import refusal_metrics
MODE=os.environ.get('BRMARL_MODE','original')
assert MODE in ('original','fixture')
FIXTURE=MODE=='fixture'
DATA_ROOT=Path(os.environ.get('BRMARL_DATA_ROOT',ROOT/'data'))
DATA={'properties':'global_properties_osrm_seconds.pkl','map':'AUG_global_map_osrm_seconds.pkl',
      'results':'epos_cache_fd.pkl','couriers':'courier_training_checkpoint_fd_seconds.pkl',
      'preferences':'glob_pref_map.pkl'}
if FIXTURE:
    assets=make_fixture_assets(ROOT/'data/fixture.json')
else:
    normalized=DATA_ROOT/'brmarl_assets.json'
    assets=load_assets(normalized,config=MarketConfig()) if normalized.exists() else load_assets(
        DATA_ROOT/DATA['map'],DATA_ROOT/DATA['preferences'],config=MarketConfig())
print('MODE:',MODE,'source:',assets['source'])
print('polygons:',len(assets['polygons']),'order columns:',sum(len(p['points']) for p in assets['polygons']))


MODE: fixture source: SYNTHETIC_ENGINEERING_FIXTURE
polygons: 6 order columns: 72


In [2]:
TRAIN_DIR=ROOT/'outputs'/MODE/'train'
CHECKPOINT=TRAIN_DIR/'checkpoint.pt'
saved=load_checkpoint(CHECKPOINT,assets)
local_cache=(ROOT/'outputs'/MODE/'routes') if FIXTURE else DATA_ROOT/'brmarl_route_cache'
ROUTE_SPEC=prepare_route_cache(assets,local_cache,fixture=FIXTURE,config=MarketConfig(**saved['market']))
N_TEST_SHIFTS=int(os.environ.get('BRMARL_TEST_SHIFTS','2'))
print('Training packets:',saved['n_training_batches'],'markets:',saved['n_training_shifts'])


Training packets: 1 markets: 12


## Полностью замороженный holdout

Новые seed, без адаптации весов, AL и threshold bank. Кэш маршрутов может дополняться,
но не передаёт агенту скрытые предпочтения или будущие исходы.

In [3]:
frozen_envs,frozen_summary=test(assets,CHECKPOINT,n_shifts=N_TEST_SHIFTS,
    output=ROOT/'outputs'/MODE/'test_frozen',adaptive=False,fixture=FIXTURE,
    route_spec=ROUTE_SPEC,shap_enabled=False,device='cpu',progress=True)
display(frozen_summary)


,shift,courier,completed,work_s,delivery_profit,return_cost,break_cost,net_profit,shift_utility,shaped_reward_sum,end_s,overtime_s,final_status
0,1012,0,31,4754.392274,10342.900444,11.643562,101.662300,10229.594582,10229.594582,10234.298879,43283.833649,83.833649,home
1,1012,1,34,5431.822063,10530.591265,14.811329,83.089663,10432.690272,10432.690272,10431.506657,43306.641571,106.641571,home
2,1012,2,36,5715.062646,13098.797740,3.090745,104.140923,12991.566072,12991.566072,12971.171953,43222.253364,22.253364,home
3,1012,3,25,3711.572715,7468.749244,18.733842,93.197976,7356.817427,7356.817427,7392.005114,43432.731783,232.731783,home
4,1012,4,15,3153.944282,5014.685758,18.387718,109.413029,4886.885012,4886.885012,4983.387657,43332.391571,132.391571,home
5,1012,5,34,5865.000653,11897.813038,13.182616,86.648429,11797.981993,11797.981993,11792.491829,43294.914837,94.914837,home
6,1012,6,27,3187.144439,7908.124403,17.911989,39.824467,7850.387947,7850.387947,7848.728425,43328.966319,128.966319,home
7,1012,7,31,4105.660283,10681.716253,43.246545,66.561785,10571.907923,10571.907923,10571.330607,43511.375123,311.375123,home
8,1012,8,32,4672.115470,10258.172491,17.561417,82.137760,10158.473315,10158.473315,10175.915168,43326.442201,126.442201,home
9,1012,9,31,4554.596426,9405.337660,15.200987,95.665029,9294.471644,9294.471644,9315.860433,43309.447109,109.447109,home


## Отдельный prequential online-тест

Здесь BRMARL обновляется после раскрытия доступной обратной связи, без доступа к другим тестовым мирам.
Это другой протокол, чем offline packet training. Начальные seed совпадают с frozen; после разных решений
эндогенные траектории могут расходиться. Начальный checkpoint загружается заново.

In [4]:
RUN_ONLINE_TEST=True
if RUN_ONLINE_TEST:
    online_envs,online_summary=test(assets,CHECKPOINT,n_shifts=N_TEST_SHIFTS,
        output=ROOT/'outputs'/MODE/'test_online',adaptive=True,fixture=FIXTURE,
        route_spec=ROUTE_SPEC,shap_enabled=True,device='cpu',progress=True)
    display(online_summary)


,shift,courier,completed,work_s,delivery_profit,return_cost,break_cost,net_profit,shift_utility,shaped_reward_sum,end_s,overtime_s,final_status
0,1012,0,31,4754.392274,10342.900444,11.643562,101.662300,10229.594582,10229.594582,10234.298879,43283.833649,83.833649,home
1,1012,1,34,5431.822063,10530.591265,14.811329,83.089663,10432.690272,10432.690272,10431.506657,43306.641571,106.641571,home
2,1012,2,36,5715.062646,13098.797740,3.090745,104.140923,12991.566072,12991.566072,12971.171953,43222.253364,22.253364,home
3,1012,3,25,3711.572715,7468.749244,18.733842,93.197976,7356.817427,7356.817427,7392.005114,43432.731783,232.731783,home
4,1012,4,15,3153.944282,5014.685758,18.387718,109.413029,4886.885012,4886.885012,4983.387657,43332.391571,132.391571,home
5,1012,5,34,5865.000653,11897.813038,13.182616,86.648429,11797.981993,11797.981993,11792.491829,43294.914837,94.914837,home
6,1012,6,27,3187.144439,7908.124403,17.911989,39.824467,7850.387947,7850.387947,7848.728425,43328.966319,128.966319,home
7,1012,7,31,4105.660283,10681.716253,43.246545,66.561785,10571.907923,10571.907923,10571.330607,43511.375123,311.375123,home
8,1012,8,32,4672.115470,10258.172491,17.561417,82.137760,10158.473315,10158.473315,10175.915168,43326.442201,126.442201,home
9,1012,9,31,4554.596426,9405.337660,15.200987,95.665029,9294.471644,9294.471644,9315.860433,43309.447109,109.447109,home


In [5]:
metrics=[]
for mode,envs in [('frozen',frozen_envs)]+([('online',online_envs)] if RUN_ONLINE_TEST else []):
    for k,env in enumerate(envs):metrics.append(dict(mode=mode,shift=k,**refusal_metrics(env)))
metrics=pd.DataFrame(metrics)
display(metrics.drop(columns=['interpretation'],errors='ignore'))
cols=['primary_affordable_policy_skips','primary_affordable_selected_other',
      'primary_affordable_price_infeasible','primary_affordable_unavailable']
display(metrics[['mode','shift']+cols])
metrics.set_index(['mode','shift'])[cols].plot.bar(figsize=(12,4));plt.tight_layout()


,mode,shift,new_orders,attempts,primary_attempts,retries,awarded,no_responses,primary_affordable_policy_skips,primary_affordable_selected_other,primary_affordable_price_infeasible,primary_affordable_unavailable,actor_updates,CFR_decisions,BRMARL_decisions,primary_no_response_with_full_feasible_courier
0,frozen,0,296,313,296,17,296,17,39,1339,0,239,10,3130,0,17
1,online,0,296,313,296,17,296,17,39,1339,0,239,10,3130,0,17


,mode,shift,primary_affordable_policy_skips,primary_affordable_selected_other,primary_affordable_price_infeasible,primary_affordable_unavailable
0,frozen,0,39,1339,0,239
1,online,0,39,1339,0,239


## Журналы, карта и SHAP

Первичный policy skip считается отдельно от занятости, выбора другого заказа и полной ценовой
неосуществимости. Маркер отказов не доказывает сговора. Карта ниже — схема точек отправления/прибытия.
Физическое перемещение внутри обязательства не моделируется.

SHAP объясняет BRMARL pricing head на конец смены при фиксированных context/latent/memory.
Если нет подходящих BRMARL-наблюдений, selection явно помечает альтернативный head probe на CFR-контекстах.
Это не объяснение gate, участия, исторических весов или причинных эффектов.

In [6]:
mode='test_online' if RUN_ONLINE_TEST else 'test_frozen'
last_dir=ROOT/'outputs'/MODE/mode/f'shift_{N_TEST_SHIFTS-1:03}'
print(json.loads((last_dir/'checks.json').read_text()))
print(json.loads((last_dir/'timings.json').read_text()))
display(IFrame(str((last_dir/'trajectory_map.html').relative_to(ROOT)),width='100%',height=500))
if (last_dir/'shap_status.json').exists():
    print(json.loads((last_dir/'shap_status.json').read_text()))
    if (last_dir/'shap_price.csv').exists():display(pd.read_csv(last_dir/'shap_price.csv').head(20))
display(pd.read_csv(last_dir/'awards.csv').head(20))


{'status': 'passed', 'invariants': ['single_award', 'non_overlapping_execution', 'no_busy_real_bids', 'one_real_bid_per_batch', 'price_bounds', 'order_cap', 'return_home', 'financial_ledger']}
{'costs_inclusive': {'total_s': 0.12456731300244428, 'calls': 164, 'p50_s': 0.0006806289998166903, 'p95_s': 0.0013923117996455403}, 'tender_inclusive': {'total_s': 20.611956012002338, 'calls': 164, 'p50_s': 0.1312199479998526, 'p95_s': 0.20805741210010636}, 'drift_inclusive': {'total_s': 0.013906623998991563, 'calls': 30, 'p50_s': 0.0004098429999430664, 'p95_s': 0.0008447145998161416}, 'simulation_wall': {'total_s': 20.66325889700056, 'calls': 1, 'p50_s': 20.66325889700056, 'p95_s': 20.66325889700056}, 'finalize_wall': {'total_s': 0.13711314600004698, 'calls': 1, 'p50_s': 0.13711314600004698, 'p95_s': 0.13711314600004698}}
{'status': 'computed', 'courier': 0, 'explained_rows': 6, 'max_additivity_error': 0.0, 'target': 'end_of_shift_BRMARL_expected_bid_ratio_at_fixed_historical_context_and_memory'

,courier,decision_row,feature,value,shap,base,prediction,additivity_error
0,0,3070,budget,0.550256,2.301983e-07,0.558888,0.558889,0.0
1,0,3070,route_cost_ratio,0.022260,-2.738289e-08,0.558888,0.558889,0.0
2,0,3070,travel_fraction,0.033072,8.332686e-09,0.558888,0.558889,0.0
3,0,3070,remaining_shift,0.020833,1.392992e-05,0.558888,0.558889,0.0
4,0,3070,home_cost_ratio,0.017847,-2.840920e-09,0.558888,0.558889,0.0
5,0,3070,completed_fraction,1.450000,-1.422331e-05,0.558888,0.558889,0.0
6,0,3070,continuous_work,0.106276,-7.934393e-07,0.558888,0.558889,0.0
7,0,3070,attempt_fraction,0.000000,0.000000e+00,0.558888,0.558889,0.0
8,0,3070,estimated_competitors,0.900000,0.000000e+00,0.558888,0.558889,0.0
9,0,3070,last_opponent_distance,0.002276,4.194804e-10,0.558888,0.558889,0.0


,shift,round,order_id,attempt,courier,start_s,finish_s,bid,logistics_cost,fee,monetary_profit,shaped_reward,execution_s,travel_s,home_cost_estimate
0,1012,1,1012:0,0,5,300.0,546.732880,848.384413,34.268456,0.0,814.115958,813.910822,246.732880,246.732880,0.205136
1,1012,1,1012:1,0,2,300.0,428.043489,217.164355,17.783818,0.0,199.380537,199.280451,128.043489,128.043489,0.100086
2,1012,1,1012:2,0,6,300.0,387.309916,475.494988,12.126377,0.0,463.368611,463.283946,87.309916,87.309916,0.084665
3,1012,2,1012:3,0,8,540.0,684.979988,514.611292,20.136109,0.0,494.475183,494.338081,144.979988,144.979988,0.137102
4,1012,2,1012:4,0,2,540.0,713.515801,238.821233,24.099417,0.0,214.721816,214.584711,173.515801,173.515801,0.137106
5,1012,2,1012:5,0,0,540.0,797.337639,294.240098,35.741339,0.0,258.498759,258.284310,257.337639,257.337639,0.214449
6,1012,3,1012:6,0,5,780.0,818.440887,648.826525,5.339012,0.0,643.487513,643.204018,38.440887,38.440887,0.283495
7,1012,3,1012:7,0,3,780.0,1009.526745,495.414963,31.878715,0.0,463.536248,463.331210,229.526745,229.526745,0.205038
8,1012,3,1012:8,0,2,780.0,805.253304,567.046920,3.507403,0.0,563.539516,563.390321,25.253304,25.253304,0.149195
9,1012,4,1012:9,0,9,1080.0,1127.910229,316.632213,6.654198,0.0,309.978015,309.932112,47.910229,47.910229,0.045903
